# Tarea 1 - Clasificación de gestos con Arduino Nano 33 BLE Sense
**Grupo:** Martinez, Reina, Torres

Repositorio: [https://github.com/miteban2000-bit/TAREA1_IABO_Martinez-Reina-Torres.git](https://github.com/miteban2000-bit/TAREA1_IABO_Martinez-Reina-Torres.git)

Este cuadernillo desarrolla los 4 puntos solicitados:
1. Carga de señales (agrupadas por clase) y cálculo de características por ventana.
2. Gráfica de dispersión con reducción de dimensionalidad (2D/3D) que separa mejor las clases.
3. División del dataset en entrenamiento y prueba.
4. Clasificador programado a partir de funciones que limitan el espacio 2D/3D.


## 0. Preparación del entorno y clonado del repositorio

In [ ]:
!git clone https://github.com/miteban2000-bit/TAREA1_IABO_Martinez-Reina-Torres.git
%cd TAREA1_IABO_Martinez-Reina-Torres
!ls -la


In [ ]:
import glob, re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, skew, entropy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis as LDA,
    QuadraticDiscriminantAnalysis as QDA,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)


## 1. Señales agrupadas por clase + archivo de características

Los datos del Arduino Nano 33 BLE Sense se recibieron por puerto serial en un
único archivo `.txt`, usando el protocolo:

```
INICIO_VENTANA <etiqueta>
ax,ay,az,gx,gy,gz
<...100 filas de datos...>
FIN_VENTANA
```

La celda siguiente busca automáticamente ese archivo dentro del repositorio
clonado (sin importar el nombre exacto), lo parsea en ventanas por clase, y
calcula las 6 características pedidas por ventana y por eje:
**media, varianza, curtosis, simetría, entropía y energía**.

In [ ]:
# Buscar el archivo de log crudo del Arduino en el repo clonado
candidatos = [f for f in glob.glob("**/*.txt", recursive=True)]
ruta_log = None
for c in candidatos:
    with open(c, encoding="utf-8", errors="replace") as fh:
        if "INICIO_VENTANA" in fh.read(2000):
            ruta_log = c
            break

print("Archivo de dataset detectado:", ruta_log)


In [ ]:
EJES = ["ax", "ay", "az", "gx", "gy", "gz"]
PATRON_FILA = re.compile(r'^-?\d+\.\d+,-?\d+\.\d+,-?\d+\.\d+,-?\d+\.\d+,-?\d+\.\d+,-?\d+\.\d+$')

def parsear_log(ruta):
    """Recorre el log crudo del Arduino y devuelve una lista de (etiqueta, DataFrame)
    por cada ventana válida. Descarta ventanas incompletas (INICIO_VENTANA sin FIN_VENTANA)."""
    ventanas = []
    etiqueta_actual, filas_actuales, en_ventana = None, [], False
    descartadas = 0

    with open(ruta, encoding="utf-8", errors="replace") as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            if linea.startswith("INICIO_VENTANA"):
                if en_ventana:
                    descartadas += 1
                partes = linea.split()
                etiqueta_actual = partes[1] if len(partes) > 1 else None
                filas_actuales = []
                en_ventana = True
            elif linea.startswith("FIN_VENTANA"):
                if en_ventana and etiqueta_actual is not None and filas_actuales:
                    ventanas.append((etiqueta_actual, pd.DataFrame(filas_actuales, columns=EJES)))
                en_ventana = False
                etiqueta_actual, filas_actuales = None, []
            elif en_ventana and PATRON_FILA.match(linea):
                filas_actuales.append([float(x) for x in linea.split(",")])
            # otras líneas (header, mensajes de estado del Arduino) se ignoran

    if descartadas:
        print(f"Aviso: se descartaron {descartadas} ventana(s) incompleta(s).")
    return ventanas

ventanas = parsear_log(ruta_log)
print(f"Ventanas válidas encontradas: {len(ventanas)}")


In [ ]:
def calcular_entropia(señal, bins=10):
    hist, _ = np.histogram(señal, bins=bins, density=True)
    hist = hist[hist > 0]
    return entropy(hist)

def calcular_energia(señal):
    return np.sum(señal ** 2) / len(señal)

def extraer_features_ventana(df_ventana):
    fila = {}
    for eje in EJES:
        s = df_ventana[eje].values
        fila[f"{eje}_media"]     = np.mean(s)
        fila[f"{eje}_varianza"]  = np.var(s)
        fila[f"{eje}_curtosis"]  = kurtosis(s)
        fila[f"{eje}_simetria"]  = skew(s)
        fila[f"{eje}_entropia"]  = calcular_entropia(s)
        fila[f"{eje}_energia"]   = calcular_energia(s)
    return fila

filas = []
for idx, (etiqueta, df_v) in enumerate(ventanas):
    fila = extraer_features_ventana(df_v)
    fila["clase"] = etiqueta
    fila["ventana_id"] = idx
    filas.append(fila)

df_features = pd.DataFrame(filas)
df_features.to_csv("caracteristicas.csv", index=False)

print(f"Ventanas procesadas: {len(df_features)} | Características por ventana: {df_features.shape[1]-2}")
print("\nConteo de ventanas por clase:")
print(df_features["clase"].value_counts())
df_features.head()


## 2. Reducción de dimensionalidad y gráfica de dispersión

Probamos dos técnicas:
- **PCA** (no supervisada): direcciones de mayor varianza.
- **LDA** (supervisada): direcciones que maximizan la separación *entre clases*.

Al ser un problema de clasificación, LDA suele mostrar una mejor separación
visual, ya que explícitamente busca esa separación (a diferencia de PCA).

In [ ]:
X = df_features.drop(columns=["clase", "ventana_id"]).values
y = df_features["clase"].values
clases = sorted(np.unique(y))

X_std = StandardScaler().fit_transform(X)


In [ ]:
def graficar_2d(X_proy, y, titulo, ejes_lbl):
    plt.figure(figsize=(7, 6))
    for c in clases:
        m = y == c
        plt.scatter(X_proy[m, 0], X_proy[m, 1], label=c, alpha=0.75, s=50)
    plt.xlabel(ejes_lbl[0]); plt.ylabel(ejes_lbl[1])
    plt.title(titulo); plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

# --- PCA 2D ---
pca_2d = PCA(n_components=2).fit_transform(X_std)
graficar_2d(pca_2d, y, "PCA 2D", ["PC1", "PC2"])

# --- LDA 2D (supervisado) ---
n_lda = min(len(clases) - 1, X.shape[1])
lda_2d = LDA(n_components=min(2, n_lda)).fit_transform(X_std, y)
if lda_2d.shape[1] == 1:
    lda_2d = np.column_stack([lda_2d, np.random.normal(0, 0.05, len(lda_2d))])
graficar_2d(lda_2d, y, "LDA 2D (supervisado)", ["LD1", "LD2"])


**Observación:** compara visualmente ambas gráficas. La proyección que muestre
las clases más separadas (menos solapamiento entre colores) es la que se debe
usar para justificar la elección en el punto 4. En nuestras pruebas, **LDA
separó claramente mejor** que PCA.

## 3. División en entrenamiento y prueba

Usamos 70% entrenamiento / 30% prueba, **estratificado por clase** para
mantener la proporción de cada gesto en ambos conjuntos (importante porque
el dataset está desbalanceado).

In [ ]:
# Reducimos a 2D con LDA para poder graficar las fronteras del clasificador
reductor = LDA(n_components=min(2, n_lda))
X_2d = reductor.fit_transform(X_std, y)
if X_2d.shape[1] == 1:
    X_2d = np.column_stack([X_2d, np.zeros(len(X_2d))])

X_train, X_test, y_train, y_test = train_test_split(
    X_2d, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Entrenamiento: {len(X_train)} muestras | Prueba: {len(X_test)} muestras")


## 4. Clasificador con funciones que limitan el espacio 2D

Proponemos y comparamos dos clasificadores sobre el espacio 2D reducido:

- **LDA**: fronteras **lineales** (rectas) entre clases.
- **QDA**: fronteras **cuadráticas** (curvas), que permiten límites más
  flexibles cuando las clases no son separables con una recta.

Ambas son funciones matemáticas explícitas que dividen el plano 2D en
regiones de clase -- justo lo que pide el punto 4.

In [ ]:
resultados = {}
for nombre, modelo in [("LDA (lineal)", LDA()), ("QDA (curvas)", QDA())]:
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    resultados[nombre] = (modelo, y_pred, acc)
    print(f"\n=== {nombre} | accuracy test = {acc:.3f} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Matriz de confusión:")
    print(confusion_matrix(y_test, y_pred, labels=clases))

mejor_nombre = max(resultados, key=lambda k: resultados[k][2])
clf, y_pred, _ = resultados[mejor_nombre]
print(f"\n>>> Clasificador elegido para graficar fronteras: {mejor_nombre}")


In [ ]:
# Graficar las regiones / fronteras de decisión del clasificador elegido
h = 0.05
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
Z_num = np.array([clases.index(c) for c in Z]).reshape(xx.shape)

plt.figure(figsize=(8, 7))
plt.contourf(xx, yy, Z_num, alpha=0.35, cmap=plt.cm.get_cmap("Pastel1", len(clases)))

cmap_puntos = plt.cm.get_cmap("Set1", len(clases))
for i, c in enumerate(clases):
    m_tr, m_te = y_train == c, y_test == c
    color = cmap_puntos(i)
    plt.scatter(X_train[m_tr, 0], X_train[m_tr, 1], color=color, marker="o",
                edgecolor="k", s=60, label=f"{c} (train)")
    plt.scatter(X_test[m_te, 0], X_test[m_te, 1], color=color, marker="^",
                edgecolor="k", s=90, label=f"{c} (test)")

plt.xlabel("LD1"); plt.ylabel("LD2")
plt.title(f"Fronteras de decisión -- {mejor_nombre}")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout(); plt.show()


## 5. Conclusiones

- El dataset presenta un **desbalance de clases** (muchas más ventanas de
  "reposo" que de los gestos direccionales), lo cual limita el desempeño
  del clasificador en las clases minoritarias.
- La reducción de dimensionalidad **supervisada (LDA)** separó las clases
  visualmente mejor que **PCA**, ya que optimiza directamente para ese fin.
- El clasificador con **fronteras curvas (QDA)** mejoró el reconocimiento de
  clases que se solapan con "reposo", frente a fronteras puramente lineales.
- Como trabajo futuro: equilibrar el número de ventanas por clase y agregar
  características adicionales (p. ej. magnitud del vector de aceleración)
  podría mejorar aún más la separación.